<a href="https://colab.research.google.com/github/dfreyes/m2t5/blob/main/m2t5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Título: Análisis de Redes y Sistemas de Recomendación

---

###Objetivos del Proyecto
1. Análisis de Redes:
  *  Entender y aplicar teoría de grafos para analizar los datos de
  interacciones entre usuarios y películas.
  *  Identificar nodos centrales, comunidades y otras métricas de red
  utilizando networkx.
  *  Visualizar la estructura del grafo e interpretar los resultados.
2. Sistemas de Recomendación:
  *  Implementar sistemas de recomendación colaborativa utilizando la
librería surprise.
  *  Evaluar diferentes algoritmos (SVD, KNN) y comparar métricas de
rendimiento.
  *   Entender cómo ajustar hiperparámetros para mejorar la precisión de las
recomendaciones.
---

###Fundamentos de ciencia de datos
Modulo 2 Taller 3
|   |   |
| :--- | :--- |
| **Autor:** | Ing. Diego F. Reyes Y. Mgtr. |
| **Fecha:** | Octubre 2026 |

Materiales
*  Entorno con Python instalado
*   Librerías:
      *  Networkx, matplotlib para análisis de redes
      *  Surprise, pandas, scikit-learn para sistemas de recomendación
*  Conjunto de datos: MovieLens 100K (ml-100k)
---

## Parte 1: Análisis de Redes con networkx

Paso 1: Preparación del Conjunto de Datos
*  Carga el conjunto de datos de MovieLens y extrae las interacciones
entre usuarios y películas. Crearás un grafo bipartito donde los usuarios
están conectados con las películas que han valorado.

Subbloque 1.1: Carga, Inspección Dinámica y Fusión de Datos (Pandas)

In [6]:
import os
import urllib.request
import zipfile

# 1. Definir URL y nombres de archivo
url = "https://grouplens.org"
zip_path = "ml-100k.zip"
data_dir = "ml-100k"

# 2. Descargar el archivo .zip si no existe
if not os.path.exists(zip_path) and not os.path.exists(data_dir):
    print("📥 Descargando MovieLens 100k...")
    urllib.request.urlretrieve(url, zip_path)
    print("✔️ Descarga completa.")

# 3. Descomprimir el archivo
if os.path.exists(zip_path):
    print("📦 Descomprimiendo archivos...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    os.remove(zip_path) # Limpia el archivo zip temporal
    print("✔️ Carpeta 'ml-100k' lista para usar.")
else:
    print("👍 La carpeta 'ml-100k' ya existe.")

# 4. Definir la variable de ruta para tu siguiente celda
data_path = os.path.join(data_dir, "u.data")


📦 Descomprimiendo archivos...
✔️ Carpeta 'ml-100k' lista para usar.


In [7]:
import pandas as pd

# Definición de rutas internas de la carpeta ml-100k
folder_name = "ml-100k"
data_path = f"{folder_name}/u.data"
item_path = f"{folder_name}/u.item"

print("🔄 1. Cargando matriz de interacciones (u.data)...")
column_names = ['user_id', 'movie_id', 'rating', 'timestamp']
df_ratings = pd.read_csv(data_path, sep='\t', names=column_names)
print(f"✔️ Tabla df_ratings cargada correctamente. Dimensiones: {df_ratings.shape} (filas, columnas)")
print("👀 Primeras 3 filas de df_ratings:")
display(df_ratings.head(3))

print("\n🔄 2. Cargando catálogo de películas (u.item)...")
movie_cols = ['movie_id', 'movie_title'] + [f'genre_{i}' for i in range(22)]
df_movies = pd.read_csv(item_path, sep='|', names=movie_cols, usecols=['movie_id', 'movie_title'], encoding='latin-1')
print(f"✔️ Tabla df_movies cargada correctamente. Dimensiones: {df_movies.shape} (filas, columnas)")
print("👀 Primeras 3 filas de df_movies:")
display(df_movies.head(3))

print("\n🔄 3. Fusionando tablas mediante 'movie_id' para obtener nombres legibles...")
df_merged = pd.merge(df_ratings, df_movies, on='movie_id')
print(f"✔️ Fusión completada. Dataset unificado df_merged. Dimensiones: {df_merged.shape}")
print("👀 Primeras 5 filas del Dataset Unificado final:")
display(df_merged[['user_id', 'movie_id', 'movie_title', 'rating']].head(5))


🔄 1. Cargando matriz de interacciones (u.data)...
✔️ Tabla df_ratings cargada correctamente. Dimensiones: (100000, 4) (filas, columnas)
👀 Primeras 3 filas de df_ratings:


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116



🔄 2. Cargando catálogo de películas (u.item)...
✔️ Tabla df_movies cargada correctamente. Dimensiones: (1682, 2) (filas, columnas)
👀 Primeras 3 filas de df_movies:


,movie_id,movie_title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)



🔄 3. Fusionando tablas mediante 'movie_id' para obtener nombres legibles...
✔️ Fusión completada. Dataset unificado df_merged. Dimensiones: (100000, 5)
👀 Primeras 5 filas del Dataset Unificado final:


,user_id,movie_id,movie_title,rating
0,196,242,Kolya (1996),3
1,186,302,L.A. Confidential (1997),3
2,22,377,Heavyweights (1994),1
3,244,51,Legends of the Fall (1994),2
4,166,346,Jackie Brown (1997),1


Subbloque 1.2: Definición y Verificación de Conjuntos Disjuntos (Teoría Bipartita)

In [ ]:
print("🔄 1. Identificando y aislando entidades únicas...")
# Usamos prefijo 'user_' para garantizar un espacio de nombres limpio y disjunto
lista_usuarios = [f"user_{uid}" for uid in df_merged['user_id'].unique()]
lista_peliculas = df_merged['movie_title'].unique().tolist()

num_usuarios_unicos = len(lista_usuarios)
num_peliculas_unicas = len(lista_peliculas)

print(f"✔️ Cardinalidad del conjunto Usuarios (|U|): {num_usuarios_unicos}")
print(f"✔️ Cardinalidad del conjunto Películas (|V|): {num_peliculas_unicas}")
print(f"📐 Suma teórica total de nodos esperados (|U| + |V|): {num_usuarios_unicos + num_peliculas_unicas}")

print("\n👀 Muestra de elementos en el conjunto Usuarios (U):", lista_usuarios[:5])
print("👀 Muestra de elementos en el conjunto Películas (V):", lista_peliculas[:3])


Subbloque 1.3: Construcción e Inspección Interna del Grafo (NetworkX)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

print("🔄 1. Instanciando objeto Graph e inyectando nodos con atributos de partición...")
B = nx.Graph()
B.add_nodes_from(lista_usuarios, bipartite=0)   # Capa 0: Usuarios
B.add_nodes_from(lista_peliculas, bipartite=1)  # Capa 1: Películas

print("🔄 2. Mapeando y construyendo aristas con pesos (ratings)...")
aristas = [(f"user_{row['user_id']}", row['movie_title'], {'weight': row['rating']}) for _, row in df_merged.iterrows()]
B.add_edges_from(aristas)
print(f"✔️ Aristas indexadas de forma completa.")

print("\n🔍 3. Inspección interna de la estructura del Grafo:")
print(f"¿Cumple la condición formal de grafo bipartito?: {nx.is_bipartite(B)}")
print(f"Número total de Nodos verificados en la red: {B.number_of_nodes()}")
print(f"Número total de Aristas verificadas en la red: {B.number_of_edges()}")

# ==========================================
# COMPONENTE VISUAL 1: DISTRIBUCIÓN DE RATINGS EN EL GRAFO
# ==========================================
print("\n📊 4. Generando gráfico de distribución de pesos (Ratings) inyectados...")
pesos_aristas = [d['weight'] for u, v, d in B.edges(data=True)]

plt.figure(figsize=(7, 4))
sns.countplot(x=pesos_aristas, palette="viridis", edgecolor="black")
plt.title("Distribución de Calificaciones (Pesos de Aristas) en la Red Bipartita", fontsize=12, fontweight='bold')
plt.xlabel("Rating (Peso de la Arista)", fontsize=10)
plt.ylabel("Conteo de Aristas", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# ==========================================
# COMPONENTE VISUAL 2: MICRO-GRAFO DE UNA SECCIÓN DE LA RED BIPARTITA
# ==========================================
print("\n🎨 5. Visualizando un subgrafo controlado para entender la estructura Bipartita...")
# Extraemos una muestra pequeña para que sea legible: User 1 y sus primeras 8 películas calificadas
usuario_muestra = 'user_1'
vecinos_muestra = list(B.neighbors(usuario_muestra))[:8]
nodos_subgrafo = [usuario_muestra] + vecinos_muestra

# Crear el subgrafo conceptual
subgrafo_visual = B.subgraph(nodos_subgrafo)

# Posicionamiento explícito en dos columnas (Bipartite Layout)
posiciones = {}
# El usuario de muestra a la izquierda
posiciones[usuario_muestra] = (0, 0.5)
# Las películas a la derecha distribuidas verticalmente
for idx, pelicula in enumerate(vecinos_muestra):
    posiciones[pelicula] = (1, idx / (len(vecinos_muestra) - 1))

# Configuración del lienzo
plt.figure(figsize=(10, 5))

# Dibujar nodos de usuarios (Capa 0 - Azul)
nx.draw_networkx_nodes(subgrafo_visual, posiciones, nodelist=[usuario_muestra],
                       node_color='#1f77b4', node_size=800, label='Usuarios (Capa 0)')

# Dibujar nodos de películas (Capa 1 - Naranja)
nx.draw_networkx_nodes(subgrafo_visual, posiciones, nodelist=vecinos_muestra,
                       node_color='#ff7f0e', node_size=400, label='Películas (Capa 1)')

# Dibujar las aristas con etiquetas de peso (rating)
nx.draw_networkx_edges(subgrafo_visual, posiciones, width=1.5, alpha=0.6, edge_color='gray')
etiquetas_aristas = {(u, v): d['weight'] for u, v, d in subgrafo_visual.edges(data=True)}
nx.draw_networkx_edge_labels(subgrafo_visual, posiciones, edge_labels=etiquetas_aristas, font_size=9)

# Añadir etiquetas de texto desplazadas para que sean legibles
posiciones_etiquetas = {k: (v[0] + 0.04 if v[0] == 1 else v[0] - 0.08, v[1]) for k, v in posiciones.items()}
nx.draw_networkx_labels(subgrafo_visual, posiciones_etiquetas, font_size=10, font_weight='bold', horizontalalignment='left')

plt.title("Esquema Didáctico de la Red Bipartita (Muestra: user_1)", fontsize=12, fontweight='bold')
plt.xlim(-0.2, 1.6) # Margen para los textos
plt.axis('off')
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


## Paso 2: Análisis del Grafo

Centralidad de Grado: Identifica a los usuarios más activos y las películas más populares.

2.1.1: Cálculo y Estructuración Separada de Métricas

In [ ]:
import networkx as nx
import pandas as pd

print("🔄 1. Computando matemáticamente la Centralidad de Grado global...")
# Se obtiene un diccionario con la métrica normalizada para cada nodo
centralidad_global = nx.degree_centrality(B)
print("✔️ Cálculo completado con éxito.")

print("\n🔄 2. Aislando y estructurando las métricas de la partición de Usuarios (bipartite=0)...")
datos_usuarios = [
    {
        "Usuario": nodo,
        "Centralidad_Normalizada": centralidad_global[nodo],
        "Total_Calificaciones_Emitidas": B.degree(nodo)
    }
    for nodo in B.nodes if B.nodes[nodo]['bipartite'] == 0
]
df_centralidad_usuarios = pd.DataFrame(datos_usuarios)
print(f"✔️ DataFrame 'df_centralidad_usuarios' creado. Dimensiones: {df_centralidad_usuarios.shape}")
print("👀 Vista previa de las primeras 3 filas:")
display(df_centralidad_usuarios.head(3))

print("\n🔄 3. Aislando y estructurando las métricas de la partición de Películas (bipartite=1)...")
datos_peliculas = [
    {
        "Película": nodo,
        "Centralidad_Normalizada": centralidad_global[nodo],
        "Total_Calificaciones_Recibidas": B.degree(nodo)
    }
    for nodo in B.nodes if B.nodes[nodo]['bipartite'] == 1
]
df_centralidad_peliculas = pd.DataFrame(datos_peliculas)
print(f"✔️ DataFrame 'df_centralidad_peliculas' creado. Dimensiones: {df_centralidad_peliculas.shape}")
print("👀 Vista previa de las primeras 3 filas:")
display(df_centralidad_peliculas.head(3))


 2.1.2: Resolución y Auditoría de las Tareas de Desafío 1 y 2

In [ ]:
print("=====================================================================")
print("🎯 RESOLUCIÓN ESPECÍFICA DE TAREAS DE DESAFÍO (CENTRALIDAD)")
print("=====================================================================")

# Desafío 1: Identificar los 5 usuarios que califican más películas
print("\n📌 Tarea de Desafío 1: Top 5 Usuarios que califican más películas:")
top_usuarios = df_centralidad_usuarios.sort_values(by="Total_Calificaciones_Emitidas", ascending=False).head(5)
display(top_usuarios.reset_index(drop=True))

# Desafío 2: Determinar cuáles son las películas más centrales en la red
print("\n📌 Tarea de Desafío 2: Top 5 Películas más centrales en la red (Más populares):")
top_peliculas = df_centralidad_peliculas.sort_values(by="Total_Calificaciones_Recibidas", ascending=False).head(5)
display(top_peliculas.reset_index(drop=True))

print("=====================================================================")




---



Coeficiente de Agrupamiento: Analiza el agrupamiento de usuarios y
películas.



2.2.1: Cálculo y Estructuración del Coeficiente Bipartito

In [ ]:
import networkx as nx
import pandas as pd

print("🔄 1. Calculando el Coeficiente de Agrupamiento Bipartito específico (Robins-Alexander)...")
# Nota metodológica: Pasamos el grafo para calcular la métrica basada en ciclos de 4 nodos (rectángulos)
clustering_bipartito = nx.bipartite.clustering(B)
print("✔️ Cálculo matemático completado.")

print("\n🔄 2. Estructurando el Coeficiente de Agrupamiento para el conjunto de Usuarios...")
datos_cc_usuarios = [
    {"Usuario": nodo, "Coeficiente_Agrupamiento_Bipartito": clustering_bipartito[nodo]}
    for nodo in B.nodes if B.nodes[nodo]['bipartite'] == 0
]
df_cc_usuarios = pd.DataFrame(datos_cc_usuarios)
print(f"✔️ DataFrame 'df_cc_usuarios' creado. Dimensiones: {df_cc_usuarios.shape}")

print("\n🔄 3. Estructurando el Coeficiente de Agrupamiento para el conjunto de Películas...")
datos_cc_peliculas = [
    {"Película": nodo, "Coeficiente_Agrupamiento_Bipartito": clustering_bipartito[nodo]}
    for nodo in B.nodes if B.nodes[nodo]['bipartite'] == 1
]
df_cc_peliculas = pd.DataFrame(datos_cc_peliculas)
print(f"✔️ DataFrame 'df_cc_peliculas' creado. Dimensiones: {df_cc_peliculas.shape}")


 2.2.2: Análisis Estadístico e Interpretación de Resultado

In [ ]:
print("=====================================================================")
print("📊 ANÁLISIS DESCRIPTIVO DEL COEFICIENTE DE AGRUPAMIENTO BIPARTITO")
print("=====================================================================")

# Cómputo de promedios para el análisis académico
mean_cc_users = df_cc_usuarios["Coeficiente_Agrupamiento_Bipartito"].mean()
mean_cc_movies = df_cc_peliculas["Coeficiente_Agrupamiento_Bipartito"].mean()

print(f"📈 Coeficiente de Agrupamiento promedio en USUARIOS: {mean_cc_users:.4f}")
print(f"📈 Coeficiente de Agrupamiento promedio en PELÍCULAS: {mean_cc_movies:.4f}")

print("\n👀 Muestra aleatoria de 3 usuarios y sus niveles de agrupamiento:")
display(df_cc_usuarios.sample(3, random_state=42).reset_index(drop=True))

print("\n👀 Muestra aleatoria de 3 películas y sus niveles de agrupamiento:")
display(df_cc_peliculas.sample(3, random_state=42).reset_index(drop=True))
print("=====================================================================")


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("🔄 1. Extrayendo un subgrafo representativo para la inspección visual...")
# Tomamos una muestra controlada de usuarios y sus películas vecinas para mantener el lienzo limpio
usuarios_muestra = df_centralidad_usuarios.sort_values(by="Total_Calificaciones_Emitidas", ascending=False).head(8)["Usuario"].tolist()
peliculas_vecinas = set()
for u in usuarios_muestra:
    peliculas_vecinas.update(list(B.neighbors(u))[:4])

nodos_visuales = usuarios_muestra + list(peliculas_vecinas)
G_clustering_visual = B.subgraph(nodos_visuales).copy()

print(f"✔️ Subgrafo listo. Nodos: {G_clustering_visual.number_of_nodes()} | Aristas: {G_clustering_visual.number_of_edges()}")

# 2. Configuración de colores basados en el valor real de su coeficiente de agrupamiento
valores_cc = [clustering_bipartito[nodo] for nodo in G_clustering_visual.nodes()]

# 3. Diseño del Layout y renderizado de la red
plt.figure(figsize=(11, 7))

# Distribución orgánica de nodos utilizando el algoritmo Kamada-Kawai
pos = nx.kamada_kawai_layout(G_clustering_visual)

# Dibujamos las aristas con tono suave para evitar ruido visual
nx.draw_networkx_edges(G_clustering_visual, pos, width=1.0, alpha=0.3, edge_color="gray")

# Dibujamos los nodos: su color depende de su coeficiente de agrupamiento (paleta 'plasma')
nodos = nx.draw_networkx_nodes(
    G_clustering_visual,
    pos,
    node_size=350,
    node_color=valores_cc,
    cmap=plt.cm.plasma,
    edgecolors="black",
    linewidths=0.8
)

# Añadir etiquetas de texto simplificadas
etiquetas_limpias = {nodo: (nodo if "user" in nodo else nodo[:12] + "...") for nodo in G_clustering_visual.nodes()}
nx.draw_networkx_labels(G_clustering_visual, pos, labels=etiquetas_limpias, font_size=8, font_weight="bold")

# =======================================================================
# CORRECCIÓN EXPLÍCITA DE LA BARRA DE COLOR (EVITA EL ATTRIBUTEERROR)
# =======================================================================
cbar = plt.colorbar(nodos, orientation="vertical", shrink=0.7)
# Usamos labelpad de forma correcta para separar el texto sin romper el objeto Text
cbar.set_label(
    "Intensidad del Coeficiente de Agrupamiento Bipartito",
    fontsize=10,
    fontweight="bold",
    labelpad=15
)

plt.title("Estructura de la Red Ponderada por Coeficiente de Agrupamiento\n(Nodos brillantes = Mayor tendencia a formar ciclos de co-calificación de 4 nodos)",
          fontsize=12, fontweight='bold', pad=15)
plt.axis("off")
plt.tight_layout()
plt.show()

print("✔️ Interpretación Visual Completa: El gráfico se ha renderizado y la barra de colores lateral está configurada correctamente.")




---



Detección de Comunidades: Aplica algoritmos de detección de
comunidades (e.g., Girvan-Newman)

2.3.1: Extracción del Subgrafo de Alta Densidad y Ejecución de Girvan-Newm

In [ ]:
import networkx as nx
from networkx.algorithms import community
import pandas as pd

print("🔄 1. Aislándolo y preparando un subgrafo inducido denso y ejecutable...")
# Tomamos los 12 usuarios más activos detectados en nuestro DataFrame de centralidad
top_12_usuarios = df_centralidad_usuarios.sort_values(by="Total_Calificaciones_Emitidas", ascending=False).head(12)["Usuario"].tolist()

# Extraemos las películas conectadas a estos usuarios muestra
peliculas_conectadas = set()
for usuario in top_12_usuarios:
    # Para mantener el grafo limpio y legible, limitamos a las primeras 4 películas de cada usuario activo
    peliculas_conectadas.update(list(B.neighbors(usuario))[:4])

# Combinamos la lista final de nodos para el subgrafo
nodos_comunidad = top_12_usuarios + list(peliculas_conectadas)
G_comunidades = B.subgraph(nodos_comunidad).copy()

print(f"✔️ Subgrafo inducido listo de forma segura. Nodos (V): {G_comunidades.number_of_nodes()} | Aristas (E): {G_comunidades.number_of_edges()}")

print("\n🔄 2. Ejecutando el algoritmo jerárquico divisivo de Girvan-Newman...")
# Este método calcula iterativamente el edge betweenness centrality y corta las aristas más centrales
generador_comunidades = community.girvan_newman(G_comunidades)

# Extraemos el primer nivel de corte para dividir la red en sus primeras macro-comunidades
primer_corte = next(generador_comunidades)
comunidades_lista = sorted(map(sorted, primer_corte))

print(f"✔️ Algoritmo completado. Se han detectado de forma preliminar {len(comunidades_lista)} comunidades independientes.")


 2.3.2: Estructuración, Conteo e Inspección de Nodos por Comunidad

In [3]:
print("=====================================================================")
print("📊 REPORTE E INSPECCIÓN TABULAR DE LAS COMUNIDADES DETECTADAS")
print("=====================================================================")

resumen_comunidades = []

for idx, comunidad in enumerate(comunidades_lista, start=1):
    # Separamos analíticamente por tipo de nodo inspeccionando sus atributos
    usuarios_en_grupo = [nodo for nodo in comunidad if G_comunidades.nodes[nodo].get('bipartite') == 0]
    peliculas_en_grupo = [nodo for nodo in comunidad if G_comunidades.nodes[nodo].get('bipartite') == 1]

    resumen_comunidades.append({
        "Comunidad": f"Comunidad {idx}",
        "Total Nodos": len(comunidad),
        "Total Usuarios": len(usuarios_en_grupo),
        "Total Películas": len(peliculas_en_grupo)
    })

    # Imprimimos los primeros elementos de cada comunidad en la consola para inspección rápida
    print(f"📁 Comunidad {idx}:")
    print(f"   👥 Miembros Usuarios (Muestra): {usuarios_en_grupo[:3]}")
    print(f"   🎬 Miembros Películas (Muestra): {peliculas_en_grupo[:3]}\n")

# Estructuración final en un DataFrame legible
df_reporte_comunidades = pd.DataFrame(resumen_comunidades)
print("👀 Resumen tabular de la segmentación estructural:")
display(df_reporte_comunidades)
print("=====================================================================")


📊 REPORTE E INSPECCIÓN TABULAR DE LAS COMUNIDADES DETECTADAS


NameError: name 'comunidades_lista' is not defined

*  La Comunidad 1 representa el núcleo principal de interacciones, concentrando de forma densa a 58 nodos (12 usuarios activos y 46 películas interconectadas).
*  La Comunidad 2 contiene únicamente 1 nodo isolado (la película Caro Diario (1994)), lo que indica matemáticamente que se trata de un nodo periférico cuya arista fue cortada tempranamente por el algoritmo debido a su baja densidad relacional en este subgrupo.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("🔄 1. Asignando vectores de colores según la comunidad detectada...")

# Creamos un diccionario para mapear rápidamente cada nodo con el ID de su comunidad
diccionario_comunidades = {}
for idx, comunidad in enumerate(comunidades_lista, start=1):
    for nodo in comunidad:
        diccionario_comunidades[nodo] = idx

# Definimos una lista de colores basada en la pertenencia de cada nodo en el subgrafo
# Comunidad 1 -> Verde esmeralda (Núcleo denso), Comunidad 2 -> Rojo carmín (Isolado)
mapeo_colores = []
for nodo in G_comunidades.nodes():
    id_com = diccionario_comunidades[nodo]
    if id_com == 1:
        mapeo_colores.append("#2ecc71") # Verde
    else:
        mapeo_colores.append("#e74c3c") # Rojo

print("🔄 2. Calculando distribución espacial orgánica (Spring Layout)...")
# El algoritmo distribuye los nodos según la fuerza de sus enlaces y modularidad
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_comunidades, k=0.25, seed=42)

# 3. Renderizado de componentes del Grafo
print("🎨 3. Dibujando aristas y nodos con enfoque de segmentación comunitaria...")
# Aristas con transparencia para evitar ruido visual
nx.draw_networkx_edges(G_comunidades, pos, width=0.8, alpha=0.25, edge_color="gray")

# Nodos pintados por comunidad
nx.draw_networkx_nodes(
    G_comunidades,
    pos,
    node_size=250,
    node_color=mapeo_colores,
    edgecolors="black",
    linewidths=0.7
)

# Etiquetas de texto controladas (Solo para los usuarios y la película aislada para no saturar)
etiquetas_filtradas = {
    nodo: (nodo if "user" in nodo or diccionario_comunidades[nodo] == 2 else "")
    for nodo in G_comunidades.nodes()
}
nx.draw_networkx_labels(G_comunidades, pos, labels=etiquetas_filtradas, font_size=8, font_weight="bold")

# Elementos complementarios de la leyenda académica
patch_com1 = plt.Line2D([0], [0], marker='o', color='w', label='Comunidad 1 (Núcleo Principal de Gustos)', markerfacecolor='#2ecc71', markersize=10)
patch_com2 = plt.Line2D([0], [0], marker='o', color='w', label='Comunidad 2 (Nodos Isolas / Periféricos)', markerfacecolor='#e74c3c', markersize=10)
plt.legend(handles=[patch_com1, patch_com2], loc="upper right", fontsize=10)

plt.title("Segmentación Estructural de la Red mediante Girvan-Newman\n(Visualización enfocada en Comunidades e Intermediación de Aristas)",
          fontsize=12, fontweight='bold', pad=15)
plt.axis("off")
plt.tight_layout()
plt.show()

print("✔️ Interpretación Académica: La visualización demuestra cómo el algoritmo agrupa de forma cohesiva a usuarios y películas con interacciones afines (Verde), mientras segrega aquellos elementos sin caminos redundantes en la muestra (Rojo).")


otra prueba

In [ ]:
import networkx as nx
from networkx.algorithms import community
import pandas as pd

print("🔄 1. Aislándolo y preparando un subgrafo inducido denso y ejecutable...")
# BAJAMOS A 30 PARA QUE SEA EJECUTABLE EN SEGUNDOS CON CORTES PROFUNDOS
top_30_usuarios = df_centralidad_usuarios.sort_values(by="Total_Calificaciones_Emitidas", ascending=False).head(30)["Usuario"].tolist()

# Extraemos las películas conectadas a estos usuarios muestra
peliculas_conectadas = set()
for usuario in top_30_usuarios:
    peliculas_conectadas.update(list(B.neighbors(usuario))[:4])

# Combinamos la lista final de nodos para el subgrafo
nodos_comunidad = top_30_usuarios + list(peliculas_conectadas)
G_comunidades = B.subgraph(nodos_comunidad).copy()

print(f"✔️ Subgrafo inducido listo de forma segura. Nodos (V): {G_comunidades.number_of_nodes()} | Aristas (E): {G_comunidades.number_of_edges()}")

print("\n🔄 2. Ejecutando el algoritmo jerárquico divisivo de Girvan-Newman...")
generador_comunidades = community.girvan_newman(G_comunidades)

# Buscaremos 4 o 5 grupos para ver comunidades reales
target_comunidades = 5

for comunidades in generador_comunidades:
    if len(comunidades) >= target_comunidades:
        comunidades_lista = sorted(map(sorted, comunidades))
        break

print(f"✔️ Algoritmo completado. Se han detectado de forma profunda {len(comunidades_lista)} comunidades.")



In [ ]:
print("=====================================================================")
print("📊 REPORTE E INSPECCIÓN TABULAR DE LAS COMUNIDADES DETECTADAS")
print("=====================================================================")

resumen_comunidades = []

for idx, comunidad in enumerate(comunidades_lista, start=1):
    # Separamos analíticamente por tipo de nodo inspeccionando sus atributos
    usuarios_en_grupo = [nodo for nodo in comunidad if G_comunidades.nodes[nodo].get('bipartite') == 0]
    peliculas_en_grupo = [nodo for nodo in comunidad if G_comunidades.nodes[nodo].get('bipartite') == 1]

    resumen_comunidades.append({
        "Comunidad": f"Comunidad {idx}",
        "Total Nodos": len(comunidad),
        "Total Usuarios": len(usuarios_en_grupo),
        "Total Películas": len(peliculas_en_grupo)
    })

    # Imprimimos los primeros elementos de cada comunidad en la consola para inspección rápida
    print(f"📁 Comunidad {idx}:")
    print(f"   👥 Miembros Usuarios (Muestra): {usuarios_en_grupo[:3]}")
    print(f"   🎬 Miembros Películas (Muestra): {peliculas_en_grupo[:3]}\n")

# Estructuración final en un DataFrame legible
df_reporte_comunidades = pd.DataFrame(resumen_comunidades)
print("👀 Resumen tabular de la segmentación estructural:")
display(df_reporte_comunidades)
print("=====================================================================")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

print("🔄 1. Asignando vectores de colores de forma genérica según comunidades...")

# 1. Creamos un diccionario para mapear rápidamente cada nodo con el ID de su comunidad
diccionario_comunidades = {}
for idx, comunidad in enumerate(comunidades_lista, start=1):
    for nodo in comunidad:
        diccionario_comunidades[nodo] = idx

# 2. Obtenemos el número total de comunidades detectadas
total_comunidades = len(comunidades_lista)

# 3. Generamos una paleta de colores automática basada en la cantidad de comunidades
# Usamos 'tab10' o 'rainbow' para asegurar colores bien diferenciados
paleta = cm.get_cmap('tab10', total_comunidades)

# 4. Asignamos a cada nodo el color exacto que le corresponde genéricamente
mapeo_colores = []
for nodo in G_comunidades.nodes():
    id_com = diccionario_comunidades[nodo]
    # Extraemos el color correspondiente (normalizado entre 0 y 1)
    color_nodo = paleta((id_com - 1) / max(1, total_comunidades - 1))
    mapeo_colores.append(color_nodo)

print(f"✔️ Mapeo completado. Se generaron {total_comunidades} colores únicos automáticamente.")

print("\n🔄 2. Calculando distribución espacial orgánica (Spring Layout)...")
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G_comunidades, k=0.25, seed=42)

# 3. Renderizado de componentes del Grafo
print("🎨 3. Dibujando aristas y nodos con enfoque de segmentación comunitaria...")
# Aristas con transparencia para evitar ruido visual
nx.draw_networkx_edges(G_comunidades, pos, width=0.8, alpha=0.2, edge_color="gray")

# Nodos pintados por comunidad de forma genérica
nx.draw_networkx_nodes(
    G_comunidades,
    pos,
    node_size=250,
    node_color=mapeo_colores,
    edgecolors="black",
    linewidths=0.7
)

# Etiquetas de texto controladas: Muestra los usuarios y los nodos que NO pertenecen a la Comunidad 1 (islas)
etiquetas_filtradas = {
    nodo: (nodo if "user" in nodo or diccionario_comunidades[nodo] != 1 else "")
    for nodo in G_comunidades.nodes()
}
nx.draw_networkx_labels(G_comunidades, pos, labels=etiquetas_filtradas, font_size=8, font_weight="bold")

# 4. Generación AUTOMÁTICA y genérica de los elementos de la leyenda
elementos_leyenda = []
for i in range(1, total_comunidades + 1):
    color_leyenda = paleta((i - 1) / max(1, total_comunidades - 1))
    # Para que sea descriptivo, identificamos la comunidad principal
    label_texto = f'Comunidad {i} (Núcleo Principal)' if i == 1 else f'Comunidad {i} (Periférica / Isla)'

    patch = plt.Line2D([0], [0], marker='o', color='w',
                       label=label_texto,
                       markerfacecolor=color_leyenda, markersize=10)
    elementos_leyenda.append(patch)

plt.legend(handles=elementos_leyenda, loc="upper right", fontsize=10)

plt.title("Segmentación Estructural de la Red mediante Girvan-Newman\n(Visualización Genérica Basada en Modularidad y Comunidades)",
          fontsize=12, fontweight='bold', pad=15)
plt.axis("off")
plt.tight_layout()
plt.show()

print("✔️ Interpretación Académica: La visualización demuestra la fragmentación de la red. Los colores distribuidos de forma automática identifican cada subgrupo independiente descubierto por el algoritmo.")


La solución rápida (Cambiar de Algoritmo): Usar el algoritmo de Louvain (nx.community.louvain_communities). Louvain no corta aristas desde afuera; busca maximizar la "densidad interna" de los grupos. Separará a tus usuarios y películas en comunidades equilibradas en menos de un segundo, sin importar qué tan densa sea la red.



---



## Parte 2: Sistema de Recomendación con surprise

Paso 1: Preparación del Conjunto de Datos

 3.1.1: Inicialización del Lector y Carga Nativa en surprise

In [ ]:
# Instalar la librería oficial de sistemas de recomendación
!pip install scikit-surprise


In [ ]:
from surprise import Reader, Dataset

print("🔄 1. Configurando los límites del dominio de datos (Rating Scale)...")
# MovieLens maneja calificaciones enteras estrictamente entre 1 y 5
lector_config = Reader(rating_scale=(1, 5))

print("\n🔄 2. Transformando df_ratings al formato formal de la librería 'surprise'...")
# Es indispensable pasar las columnas en el orden estricto: [User, Item, Rating]
datos_surprise = Dataset.load_from_df(df_ratings[['user_id', 'movie_id', 'rating']], lector_config)

print("✔️ Transformación exitosa.")
print(f"📌 Tipo de objeto generado: {type(datos_surprise)}")
print("💡 Nota: Los objetos 'Dataset' en surprise son estructuras optimizadas listas para el entrenamiento y validación cruzada.")


3.1.2: Control de Calidad y División de Datos (Train/Test Split)

In [ ]:
from surprise.model_selection import train_test_split

print("🔄 1. Ejecutando segmentación de datos (80% Entrenamiento / 20% Prueba)...")
# Fijamos una semilla (random_state) para garantizar la reproducibilidad de los experimentos de la maestría
trainset, testset = train_test_split(datos_surprise, test_size=0.2, random_state=42)

print("\n=======================================================")
print("📊 REPORTE DE CONTROL DE CALIDAD - APRENDIZAJE")
print("=======================================================")
print(f"📈 Número de interacciones en el conjunto de Entrenamiento (Trainset): {trainset.n_ratings}")
print(f"📉 Número de interacciones en el conjunto de Evaluación (Testset): {len(testset)}")
print(f"👥 Usuarios únicos indexados en el entrenamiento: {trainset.n_users}")
print(f"🎬 Películas únicas indexadas en el entrenamiento: {trainset.n_items}")
print("=======================================================")

Paso 2: Entrenamiento del Modelo

3.2.1: Inicialización, Entrenamiento y Evaluación de Métricas Básicas (SVD vs KNN)

In [ ]:
from surprise import SVD, KNNBasic
from surprise import accuracy

print("🔄 1. Inicializando los algoritmos con configuraciones base...")
modelo_svd = SVD(random_state=42) # Fijamos semilla para reproducibilidad
modelo_knn = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}) # KNN basado en usuarios con similitud del coseno

print("\n🏋️ 2. Entrenando el algoritmo SVD sobre las 80,000 interacciones...")
modelo_svd.fit(trainset)
print("🔮 Generando predicciones con SVD para el Testset...")
predicciones_svd = modelo_svd.test(testset)

print("\n🏋️ 3. Entrenando el algoritmo KNNBasic sobre las 80,000 interacciones...")
modelo_knn.fit(trainset)
print("🔮 Generando predicciones con KNN para el Testset...")
predicciones_knn = modelo_knn.test(testset)

print("\n=======================================================")
print("📊 RESULTADOS COMPARATIVOS DE PRECISIÓN BASE")
print("=======================================================")
print("📉 MÉTRICAS PARA SVD:")
rmse_svd = accuracy.rmse(predicciones_svd, verbose=False)
mae_svd = accuracy.mae(predicciones_svd, verbose=False)
print(f"   ✔️ RMSE SVD: {rmse_svd:.4f}")
print(f"   ✔️ MAE SVD:  {mae_svd:.4f}")

print("\n📉 MÉTRICAS PARA KNN BASIC:")
rmse_knn = accuracy.rmse(predicciones_knn, verbose=False)
mae_knn = accuracy.mae(predicciones_knn, verbose=False)
print(f"   ✔️ RMSE KNN: {rmse_knn:.4f}")
print(f"   ✔️ MAE KNN:  {mae_knn:.4f}")
print("=======================================================")


In [ ]:
import pandas as pd

print("📊 Estructurando tabla comparativa de desempeño de los algoritmos...")

# Crear el diccionario con las métricas obtenidas en la celda anterior
datos_comparacion = {
    "Algoritmo": ["SVD (Factorización de Matrices)", "KNN Basic (Basado en Vecindarios)"],
    "RMSE (Menor es mejor)": [rmse_svd, rmse_knn],
    "MAE (Menor es mejor)": [mae_svd, mae_knn]
}

# Construcción formal del DataFrame de control
df_metricas_base = pd.DataFrame(datos_comparacion)
display(df_metricas_base.reset_index(drop=True))

print("\n📝 Interpretación Preliminar: El algoritmo que presente el RMSE más bajo posee una mayor capacidad para estimar con precisión las calificaciones no observadas de los usuarios.")


3.2.3: Visualización Gráfica del Rendimiento Base (SVD vs KNN)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Generando gráfico comparativo de métricas de precisión (RMSE y MAE)...")

# Reestructurar el DataFrame a formato largo (melt) para facilitar el graficado con Seaborn
df_melted = df_metricas_base.melt(
    id_vars="Algoritmo",
    value_vars=["RMSE (Menor es mejor)", "MAE (Menor es mejor)"],
    var_name="Métrica",
    value_name="Valor"
)

# Configuración del lienzo
plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x="Métrica",
    y="Valor",
    hue="Algoritmo",
    data=df_melted,
    palette=["#2ecc71", "#3498db"], # Verde para SVD y Azul para KNN
    edgecolor="black",
    linewidth=1.2
)

# Añadir etiquetas de valor numérico sobre cada barra para máxima precisión visual
for p in ax.patches:
    if p.get_height() > 0: # Evitar etiquetas en barras vacías si las hay
        ax.annotate(
            f"{p.get_height():.4f}",
            (p.get_x() + p.get_width() / 2., p.get_height()),
            ha='center',
            va='center',
            xytext=(0, 8),
            textcoords='offset points',
            fontsize=10,
            fontweight='bold'
        )

# Configuración de títulos y ejes descriptivos
plt.title("Comparativa de Error Base: SVD vs KNN Basic", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Magnitud del Error", fontsize=10)
plt.xlabel("", fontsize=10)
plt.ylim(0, max(rmse_knn, rmse_svd) * 1.2) # Dar margen superior para las etiquetas
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc="upper right", title="Algoritmos")

sns.despine()
plt.tight_layout()
plt.show()

print(f"✔️ Análisis Gráfico Completo. Confirmamos visualmente que SVD reduce el RMSE en {rmse_knn - rmse_svd:.4f} puntos frente a KNN.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from surprise import SVD, KNNBasic, Reader, Dataset
from surprise.model_selection import train_test_split

print("=====================================================================")
print("🚀 CONTROL LOCAL: PROCESAMIENTO INMUNE Y GRÁFICO TRIPLE COMPLETO")
print("=====================================================================")

# 1. Recuperar directamente el archivo físico original para limpiar la memoria
folder_name = "ml-100k"
data_path_original = f"{folder_name}/u.data"
column_names_rec = ['user_id', 'movie_id', 'rating', 'timestamp']

# Leemos de forma local e independiente de celdas superiores
df_ratings_local = pd.read_csv(data_path_original, sep='\t', names=column_names_rec)

# Cargar en Surprise y forzar división limpia 80/20
lector_local = Reader(rating_scale=(1, 5))
datos_local_surprise = Dataset.load_from_df(df_ratings_local[['user_id', 'movie_id', 'rating']], lector_local)
trainset_local, testset_local = train_test_split(datos_local_surprise, test_size=0.2, random_state=42)

print("🔮 Entrenando SVD y KNN Basic locales con los 100,000 registros...")
modelo_svd_local = SVD(random_state=42).fit(trainset_local)
modelo_knn_local = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}, verbose=False).fit(trainset_local)

predicciones_svd_locales = modelo_svd_local.test(testset_local)
predicciones_knn_locales = modelo_knn_local.test(testset_local)

# 2. Selección automática de un usuario real con datos densos
dict_titulos = dict(zip(df_movies['movie_id'].astype(int), df_movies['movie_title']))
usuarios_test_local = list(set(p.uid for p in predicciones_svd_locales))

usuario_seleccionado = None
pred_u_svd = []

for u in sorted(usuarios_test_local):
    filtro_u = [p for p in predicciones_svd_locales if p.uid == u]
    if len(filtro_u) >= 5:
        usuario_seleccionado = u
        pred_u_svd = filtro_u
        break

print(f"✔️ Usuario denso identificado automáticamente: {usuario_seleccionado}")

# Mapear predicciones del modelo KNN para este mismo usuario
pred_u_knn = [p for p in predicciones_knn_locales if p.uid == usuario_seleccionado]
dict_knn_est = {p.iid: p.est for p in pred_u_knn}

datos_grafico = []
# Extraer exactamente 5 películas consecutivas del testset para este usuario
for p in pred_u_svd[:5]:
    id_p_int = int(p.iid)
    titulo_original = dict_titulos.get(id_p_int, f"ID: {id_p_int}")
    titulo_corto = titulo_original[:20] + "..." if len(titulo_original) > 20 else titulo_original

    datos_grafico.append({
        "Película": titulo_corto,
        "Calificación Real": float(p.r_ui),
        "Predicción SVD": float(p.est),
        "Predicción KNN": float(dict_knn_est.get(p.iid, trainset_local.global_mean))
    })

df_plot_triple = pd.DataFrame(datos_grafico)

# 3. RENDERIZADO DEL GRÁFICO COMPARATIVO COMPLETO (5 MUESTRAS MULTI-BARRAS)
print("\n🎨 Generando lienzo con la comparativa de 5 películas...")
x = np.arange(len(df_plot_triple))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5.5))

# Dibujar las tres barras por película
barras_reales = ax.bar(x - width, df_plot_triple["Calificación Real"], width, label='Calificación Real (Usuario)', color='#7f8c8d', edgecolor='black')
barras_svd = ax.bar(x, df_plot_triple["Predicción SVD"], width, label='Predicción SVD (Matriz Latente)', color='#2ecc71', edgecolor='black')
barras_knn = ax.bar(x + width, df_plot_triple["Predicción KNN"], width, label='Predicción KNN (Vecindarios)', color='#3498db', edgecolor='black')

def agregar_etiquetas(barras):
    for bar in barras:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

agregar_etiquetas(barras_reales)
agregar_etiquetas(barras_svd)
agregar_etiquetas(barras_knn)

# Configuración formal del gráfico
ax.set_ylabel('Estrellas (Escala de 1 a 5)', fontsize=10, fontweight='bold')
ax.set_title(f'Evaluación del Modelo en Acción: Humano vs SVD vs KNN Basic\n(Muestra de 5 Películas Reales - Usuario de Control: {usuario_seleccionado})', fontsize=12, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(df_plot_triple["Película"], rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 5.9)
ax.legend(loc="upper right", title="Entidades de Evaluación")
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()




---



Paso 3: Ajuste de Hiperparámetros

3.3.1: Configuración y Ejecución del Grid Search (Optimización de SVD)

In [ ]:
from surprise.model_selection import GridSearchCV
from surprise import SVD

print("🔄 1. Definiendo la malla de hiperparámetros para el modelo SVD usando rangos...")

# Creamos las listas usando funciones nativas para evitar problemas de formato con los corchetes
lista_factores = list(range(20, 51, 30))  # Esto genera la lista con 20 y 50
lista_epocas = list(range(10, 21, 10))    # Esto genera la lista con 10 y 20
lista_lr = list([0.002, 0.005])

param_grid = {
    'n_factors': lista_factores,   # Cantidad de variables latentes ocultas (20, 50)
    'n_epochs': lista_epocas,      # Iteraciones de optimización (10, 20)
    'lr_all': lista_lr             # Tasas de aprendizaje (0.002, 0.005)
}

print(f"⚙️ Malla a evaluar: Factors={lista_factores} | Epochs={lista_epocas} | LR={lista_lr}")

print("\n🔄 2. Inicializando GridSearchCV con validación cruzada de 3 pliegues (CV=3)...")
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1, joblib_verbose=1)

print("\n🏋️ 3. Ejecutando la búsqueda exhaustiva en cuadrícula sobre el dataset...")
gs.fit(datos_surprise)

print("\n=======================================================")
print("🎯 RESULTADOS DE OPTIMIZACIÓN ENCONTRADOS")
print("=======================================================")
print(f"📈 Mejor puntuación RMSE obtenida en Validación Cruzada: {gs.best_score['rmse']:.4f}")
print(f"⚙️ Configuración óptima de parámetros: {gs.best_params['rmse']}")
print("=======================================================")



In [ ]:
from surprise.model_selection import GridSearchCV
from surprise import SVD

print("🔄 1. Definiendo una malla expandida y regularizada para optimizar el SVD...")

# Explicación del cambio de rangos:
# - Bajamos 'n_factors' porque sistemas con muchos usuarios/películas funcionan mejor con menos dimensiones conceptuales.
# - Subimos 'n_epochs' dando más margen pero con un paso fino.
# - Agregamos 'reg_all' (Regularización) para controlar el castigo a valores extremos.
lista_factores = [5, 10, 20]
lista_epocas = [10, 15, 25]
lista_lr = [0.005, 0.007]
lista_reg = [0.04, 0.1] # <--- ¡LA CLAVE! Valores por defecto suelen quedarse cortos

param_grid = {
    'n_factors': lista_factores,   # Explora estructuras más simples y robustas
    'n_epochs': lista_epocas,      # Da más opciones de convergencia
    'lr_all': lista_lr,            # Tasas ajustadas al estándar de SVD
    'reg_all': lista_reg           # Evita que el modelo memorice ruido
}

print(f"⚙️ Nueva Malla a evaluar: Factors={lista_factores} | Epochs={lista_epocas} | LR={lista_lr} | Reg={lista_reg}")

print("\n🔄 2. Inicializando GridSearchCV con validación cruzada robusta (CV=5)...")
# Subimos a cv=5 para entrenar con el 80% de los datos y evaluar mejor
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=-1, joblib_verbose=1)

print("\n🏋️ 3. Ejecutando la búsqueda científica en cuadrícula sobre el dataset...")
gs.fit(datos_surprise)

print("\n=======================================================")
print("🎯 RESULTADOS DE OPTIMIZACIÓN ENCONTRADOS")
print("=======================================================")
print(f"📈 Mejor puntuación RMSE obtenida en Validación Cruzada: {gs.best_score['rmse']:.4f}")
print(f"⚙️ Configuración óptima de parámetros: {gs.best_params['rmse']}")
print("=======================================================")


In [ ]:
from surprise.model_selection import GridSearchCV
from surprise import SVD, Reader, Dataset

# === PASO CRÍTICO: Asegura la escala real de tus datos ===
# Cambia 'tu_dataframe_transaccional' por el nombre de tu tabla de 100k filas
min_voto = df_merged['rating'].min()
max_voto = df_merged['rating'].max()
print(f"📊 Ajustando escala del recomendador a la realidad: de {min_voto} a {max_voto} estrellas.")

reader = Reader(rating_scale=(min_voto, max_voto))
datos_surprise = Dataset.load_from_df(df_merged[['user_id', 'movie_id', 'rating']], reader)
# ========================================================

print("\n🔄 1. Definiendo la nueva malla basada en las necesidades del SVD...")
param_grid = {
    'n_factors': [20, 35, 50],    # Subimos el rango porque 20 se quedó corto
    'n_epochs': [15, 20, 25],     # Quitamos el 10 para obligar al SVD a converger mejor
    'lr_all': [0.005, 0.007],     # Mantenemos las tasas estándar
    'reg_all': [0.1, 0.15, 0.2]   # Expandimos hacia arriba porque el modelo pidió más regularización
}

print("\n🔄 2. Inicializando GridSearchCV con CV=5...")
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=5, n_jobs=-1, joblib_verbose=1)

print("\n🏋️ 3. Ejecutando optimización...")
gs.fit(datos_surprise)

print("\n=======================================================")
print("🎯 NUEVOS RESULTADOS ENCONTRADOS")
print("=======================================================")
print(f"📈 Mejor puntuación RMSE: {gs.best_score['rmse']:.4f}")
print(f"⚙️ Parámetros óptimos: {gs.best_params['rmse']}")
print("=======================================================")


Top 5 Recomendaciones Personalizadas

3.4.1: Entrenamiento Definitivo del Modelo SVD Sintonizado

In [ ]:
from surprise import SVD

print("🔄 1. Instanciando el modelo SVD sintonizado con tus parámetros ganadores...")
# Configuramos el modelo final con los hiperparámetros de tu imagen
modelo_svd_final = SVD(
    n_factors=50,
    n_epochs=25,
    lr_all=0.007,
    reg_all=0.1,
    random_state=42
)

print("\n🏋️ 2. Entrenando el modelo definitivo sobre el Trainset de control (80,000 ratings)...")
modelo_svd_final.fit(trainset)
print("✔️ Modelo entrenado y listo para generar recomendaciones personalizadas.")


3.4.2:  Generación del Top 5 de Recomendaciones Personalizadas (Películas No Vistas)

In [ ]:
import pandas as pd
from surprise import SVD, Reader, Dataset

print("=====================================================================")
print("🚀 RECUPERACIÓN COMPLETA DE 100K REGISTROS Y TOP 5 DEFINITIVO")
print("=====================================================================")

# 1. Volver a leer directamente el archivo físico original para recuperar las 100k filas
folder_name = "ml-100k"
data_path_original = f"{folder_name}/u.data"

column_names_rec = ['user_id', 'movie_id', 'rating', 'timestamp']
# Forzamos la lectura limpia del archivo original en una variable local nueva e inmune a recortes
df_ratings_completo = pd.read_csv(data_path_original, sep='\t', names=column_names_rec)

print(f"📊 Control de Calidad del Dataset Original Físico:")
print(f"   ✔️ Total filas recuperadas del archivo u.data: {df_ratings_completo.shape[0]}")

# 2. Configurar el lector y cargar en el formato nativo de Surprise
lector_completo = Reader(rating_scale=(1, 5))
datos_surprise_reales = Dataset.load_from_df(df_ratings_completo[['user_id', 'movie_id', 'rating']], lector_completo)

# Construir el conjunto de entrenamiento global con el universo completo de datos
trainset_real = datos_surprise_reales.build_full_trainset()
print(f"   ✔️ Total de interacciones en la memoria de Surprise: {trainset_real.n_ratings}")
print(f"   ✔️ Total de usuarios en la memoria de Surprise: {trainset_real.n_users}")
print(f"   ✔️ Total de películas en la memoria de Surprise: {trainset_real.n_items}")

# 3. Entrenamiento definitivo del algoritmo SVD Optimizado
modelo_svd_final = SVD(
    n_factors=50,
    n_epochs=25,
    lr_all=0.007,
    reg_all=0.1,
    random_state=42
)
print("\n🏋️ Entrenando SVD sintonizado sobre el dataset completo...")
modelo_svd_final.fit(trainset_real)
print("✔️ ¡Entrenamiento completado de forma exitosa!")

# 4. Definición del Usuario Objetivo del Taller (ID numérico: 13)
id_usuario_taller = 13

# Extraer el historial de consumo real buscando directamente en la tabla completa
# En u.data los IDs son estrictamente enteros (int), lo que elimina problemas de texto
peliculas_ya_vistas = set(df_ratings_completo[df_ratings_completo['user_id'] == id_usuario_taller]['movie_id'].tolist())

# Obtener la lista de todas las películas que el modelo conoce a nivel global
todas_las_peliculas_modelo = [trainset_real.to_raw_iid(i) for i in trainset_real.all_items()]

# Filtrar las candidatas (las que el usuario 13 NO ha visto jamás)
peliculas_candidatas_reales = [mid for mid in todas_las_peliculas_modelo if mid not in peliculas_ya_vistas]

print(f"\n👤 Perfil del Usuario Analizado: {id_usuario_taller}")
print(f"   ✔️ Películas consumidas en su historial real: {len(peliculas_ya_vistas)}")
print(f"   ✔️ Películas nuevas candidatas para predecir: {len(peliculas_candidatas_reales)}")

# 5. Cómputo de calificaciones estimadas personalizadas
print(f"\n🔮 Calculando predicciones personalizadas de alta fidelidad...")
resultados_predicciones = []

for mid in peliculas_candidatas_reales:
    pred = modelo_svd_final.predict(id_usuario_taller, int(mid))
    resultados_predicciones.append({
        "movie_id": int(mid),
        "Calificación_Estimada": float(pred.est)
    })

df_pred = pd.DataFrame(resultados_predicciones)

# 6. Fusión impecable con df_movies asegurando consistencia de tipos numéricos (int)
df_movies['movie_id'] = df_movies['movie_id'].astype(int)
df_pred['movie_id'] = df_pred['movie_id'].astype(int)

df_final_rec = pd.merge(df_pred, df_movies[['movie_id', 'movie_title']], on='movie_id')

# Extraer el Top 5 definitivo ordenado de mayor a menor preferencia estimada
top_5_taller_final = df_final_rec.sort_values(by="Calificación_Estimada", ascending=False).head(5)

print("\n=====================================================================")
print(f"🎯 TAREA DESAFÍO FINAL: TOP 5 RECOMENDACIONES PERSONALIZADAS (USER {id_usuario_taller})")
print("=====================================================================")
display(top_5_taller_final[['movie_title', 'Calificación_Estimada']].reset_index(drop=True))
print("=====================================================================")


https://www.google.com/search?q=ia&client=firefox-b-d&hs=y3RB&sca_esv=26f62205f8085015&sxsrf=APpeQnts7kA_gAjkQlFuqb82o7ASh-j7nQ%3A1791587634186&fbs=ABfTbFUDadgeu2mn4mYJ8iEZ1GUDd8ABuXxNzQEi57SWOuuPdcURz3vM2j0dknRjZr5tESjRZlWxtHZPI_s9Tv5zfxnnqmeUogUhwx1VgK6zi3Q_AyeXxsDre_tqUAq4nb3CHG-5oa40JDUSUCUvspWmNlPqSNVmjwMDg1-50w4swq8IZCRRJyDIyor-pa4YyjIQSk5YRU_1eZOAIX5jn08jNAqkOBLdTw&vsint=&aep=1&ntc=1&cs=1&sa=X&ved=2ahUKEwi72JOEiK6XAxU9ElkFHa9rOhoQ2J8OegQIFhAD&biw=1522&bih=840&dpr=1.07&mstk=AUtExfAOT8y11L_LE0nMcV-kEE0o7sW1CuuXQ6RjgT5DUFn4_DtTC_TsbcQatd7lsBr7MYJmW03kVpdY4gIuSE7WkER41O2iP-LVt68flUuiYxpo1jdDsv1eb_fY2GolNyY6-xLwYwZEhSrsgX9wL9khbb9wlwNm4zxTxsAgbH8YcW_z839GYnbyXwK5k8mJoga0_dOv_VPKnNTcyiNbCnahG9VbUOlQhn_MQSPuDTlSQ83aeaUqLwhLOh6Aq4BiVGfy636NTRx1rASP6YUL6Q9qZKAnHTorer5yxqtA5eKRwOaMaQtkyCR9OqLmGcpUY45jHHyEOTr4IK0-hA&csuir=1&mtid=NHXJaoK6G8yTwt0Pw9XWiAM&lns_mode=cvst&udm=50